# E03 Readiness and Correctness Notebook

This notebook is for **Experiment E03: Modality pooling experiment**.

## Purpose

E03 is meant to answer:

> Should audio, video, and audiovisual conditions be pooled together, or modeled separately by modality?

The intended fixed controls are:

- `target = coarse_affect`
- `model = ridge`
- `cv = within_subject_loso_session`

So E03 should vary **modality policy only**.

## What this notebook does

1. Inspects the current workbook definition of E03
2. Inspects the current registry definition of E03
3. Audits modality labels in the dataset index
4. Checks class/sample sufficiency by modality, subject, and session
5. Checks for a key scientific risk: **task composition confounds across modalities**
6. Prints a practical readiness verdict
7. Recommends exactly what should be changed to make E03 stronger and cleaner


## Scientific standard for E03

E03 is scientifically strongest if:

- pooled-modality and modality-specific conditions are **explicitly pre-registered**
- the exact modality set is fixed in advance
- modality-specific runs are only interpreted if they pass sufficiency checks
- the comparison is not contaminated by hidden task-composition differences across modalities
- the workbook and executable path mean the same thing


In [6]:
from pathlib import Path
import json
import os
import sys
import subprocess

import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for base in candidates:
        if (base / "configs" / "decision_support_registry.json").exists():
            return base
    raise FileNotFoundError("Could not find repo root containing configs/decision_support_registry.json")

REPO_ROOT = find_repo_root()
INDEX_CSV = REPO_ROOT / "Data/processed/dataset_index.csv"
DATA_ROOT = REPO_ROOT / "Data"
CACHE_DIR = REPO_ROOT / "Data/processed/feature_cache"
REGISTRY = REPO_ROOT / "configs/decision_support_registry.json"

# You may need to adjust this if your workbook is stored elsewhere.
WORKBOOK = REPO_ROOT / "templates/thesis_experiment_program_revised.xlsx"
if not WORKBOOK.exists():
    alt = Path.cwd() / "thesis_experiment_program_revised.xlsx"
    if alt.exists():
        WORKBOOK = alt

print("Repo root:", REPO_ROOT)
print("Index:", INDEX_CSV, INDEX_CSV.exists())
print("Registry:", REGISTRY, REGISTRY.exists())
print("Workbook:", WORKBOOK, WORKBOOK.exists())


Repo root: D:\Khaled\Projects\VScode\Thesis
Index: D:\Khaled\Projects\VScode\Thesis\Data\processed\dataset_index.csv True
Registry: D:\Khaled\Projects\VScode\Thesis\configs\decision_support_registry.json True
Workbook: D:\Khaled\Projects\VScode\Thesis\templates\thesis_experiment_program_revised.xlsx True


## 1. Inspect the workbook definition of E03


In [7]:
exp_defs = pd.read_excel(WORKBOOK, sheet_name="Experiment_Definitions")
e03_wb = exp_defs[exp_defs["experiment_id"].astype(str).str.upper() == "E03"].copy()
e03_wb


,experiment_id,enabled,start_section,end_section,base_artifact_id,target,cv,model,subject,train_subject,test_subject,filter_task,filter_modality,reuse_policy,search_space_id
2,E03,No,dataset_selection,evaluation,NaN,coarse_affect,within_subject_loso_session,ridge,NaN,NaN,NaN,NaN,VARIES,require_explicit_base,NaN


In [8]:
master = pd.read_excel(WORKBOOK, sheet_name="Master_Experiments")
e03_master = master[master["Experiment_ID"].astype(str).str.upper() == "E03"].copy()
e03_master


,Experiment_ID,Short_Title,Category,Evidential_Role,Stage,Priority,Decision_Supported,Exact_Question,Hypothesis_or_Expectation,Manipulated_Factor,...,Failure_or_Warning_Pattern,Threats_to_Validity,Reporting_Destination,Interpretation_Boundary,Status,Owner,Outcome_Summary,Decision_Taken,Notes,Experiment_Ready
2,E03,Modality pooling experiment,Target-definition,Secondary decision-support,Stage 1 - Target lock,Medium,Modality pooling policy,"Should audio, video, and audiovisual condition...",Evaluate under leakage-aware claim-matched spl...,Pooling strategy by modality,...,"Weak-split inflation, instability, class colla...","Class imbalance, sample limits, domain shift, ...",Chapter 3 Method choice,Interpret only under stated split logic/eviden...,Planned,Khaled (thesis author),NaN,NaN,NaN,NaN


### Workbook interpretation

Things to look for immediately:

- Is `enabled` still `No`?
- Is `filter_modality` still `VARIES`?
- Is there any `search_space_id` attached?
- Does the workbook explicitly operationalize pooled vs specific modality conditions?

If `filter_modality = VARIES` and no executable expansion is attached, the workbook is still descriptive rather than runnable.


## 2. Inspect the registry definition of E03


In [9]:
registry_data = json.loads(REGISTRY.read_text(encoding="utf-8"))
e03_reg = None
for item in registry_data.get("experiments", []):
    if item.get("experiment_id") == "E03":
        e03_reg = item
        break

from pprint import pprint
pprint(e03_reg)


{'blocked_reasons': [],
 'dataset_scope': 'Internal BAS2 repeated-session index.',
 'decision_id': 'D01',
 'executable_now': True,
 'execution_status': 'executable',
 'experiment_id': 'E03',
 'fixed_controls': 'coarse_affect target, ridge model, '
                   'within_subject_loso_session split, seed=42.',
 'manipulated_factor': 'Pooling strategy by modality',
 'models': ['ridge'],
 'notes': 'Compares pooled modalities vs per-modality filtering.',
 'output_dir': 'outputs/artifacts/decision_support/E03',
 'primary_metric': 'balanced_accuracy',
 'secondary_metrics': ['macro_f1', 'accuracy'],
 'split_logic': 'within_subject_loso_session, per subject.',
 'stage': 'Stage 1 - Target lock',
 'target_definition': 'coarse_affect',
 'title': 'Modality pooling experiment',
 'variant_templates': [{'expand': {'subject': 'subjects'},
                        'params': {'cv': 'within_subject_loso_session',
                                   'model': 'ridge',
                                   't

### Registry interpretation

The current registry path is operationally useful if it does these things:

- one pooled-modality template
- one modality-specific template
- subject expansion
- fixed `coarse_affect`, `ridge`, and `within_subject_loso_session`

However, it can still be **scientifically weaker than ideal** if it auto-expands over whatever modality labels happen to exist in the dataset rather than a fixed approved modality set.


## 3. Inspect modality labels in the dataset index


In [10]:
df = pd.read_csv(INDEX_CSV)
print(df.shape)
print(df.columns.tolist())
df.head()


(3402, 17)
['sample_id', 'subject', 'session', 'bas', 'run', 'task', 'emotion', 'coarse_affect', 'modality', 'beta_index', 'beta_file', 'beta_path', 'mask_path', 'regressor_label', 'raw_label', 'subject_session', 'subject_session_bas']


,sample_id,subject,session,bas,run,task,emotion,coarse_affect,modality,beta_index,beta_file,beta_path,mask_path,regressor_label,raw_label,subject_session,subject_session_bas
0,sub-001_ses-01_BAS2_run-1_beta-1,sub-001,ses-01,BAS2,1,passive,anger,negative,audio,1,beta_0001.nii,sub-001/ses-01/BAS2/beta_0001.nii,sub-001/ses-01/BAS2/mask.nii,run-1_passive_anger_audio,run-1_passive_anger_audio,sub-001_ses-01,sub-001_ses-01_BAS2
1,sub-001_ses-01_BAS2_run-1_beta-2,sub-001,ses-01,BAS2,1,passive,anger,negative,video,2,beta_0002.nii,sub-001/ses-01/BAS2/beta_0002.nii,sub-001/ses-01/BAS2/mask.nii,run-1_passive_anger_video,run-1_passive_anger_video,sub-001_ses-01,sub-001_ses-01_BAS2
2,sub-001_ses-01_BAS2_run-1_beta-3,sub-001,ses-01,BAS2,1,passive,anger,negative,audiovisual,3,beta_0003.nii,sub-001/ses-01/BAS2/beta_0003.nii,sub-001/ses-01/BAS2/mask.nii,run-1_passive_anger_audiovisual,run-1_passive_anger_audiovisual,sub-001_ses-01,sub-001_ses-01_BAS2
3,sub-001_ses-01_BAS2_run-1_beta-4,sub-001,ses-01,BAS2,1,passive,anxiety,negative,audio,4,beta_0004.nii,sub-001/ses-01/BAS2/beta_0004.nii,sub-001/ses-01/BAS2/mask.nii,run-1_passive_anxiety_audio,run-1_passive_anxiety_audio,sub-001_ses-01,sub-001_ses-01_BAS2
4,sub-001_ses-01_BAS2_run-1_beta-5,sub-001,ses-01,BAS2,1,passive,anxiety,negative,video,5,beta_0005.nii,sub-001/ses-01/BAS2/beta_0005.nii,sub-001/ses-01/BAS2/mask.nii,run-1_passive_anxiety_video,run-1_passive_anxiety_video,sub-001_ses-01,sub-001_ses-01_BAS2


In [11]:
required_cols = ["subject", "session"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required base columns: {missing_required}")

# Find likely modality/task/target columns flexibly.
candidate_modality_cols = [c for c in df.columns if "modality" in c.lower()]
candidate_task_cols = [c for c in df.columns if "task" in c.lower()]
candidate_target_cols = [c for c in ["coarse_affect", "emotion", "binary_valence_like"] if c in df.columns]

print("Candidate modality columns:", candidate_modality_cols)
print("Candidate task columns:", candidate_task_cols)
print("Candidate target columns:", candidate_target_cols)


Candidate modality columns: ['modality']
Candidate task columns: ['task']
Candidate target columns: ['coarse_affect', 'emotion']


In [12]:
# Pick the first reasonable modality/task columns if present.
MODALITY_COL = candidate_modality_cols[0] if candidate_modality_cols else None
TASK_COL = candidate_task_cols[0] if candidate_task_cols else None
TARGET_COL = "coarse_affect" if "coarse_affect" in df.columns else (candidate_target_cols[0] if candidate_target_cols else None)

print("Chosen modality column:", MODALITY_COL)
print("Chosen task column:", TASK_COL)
print("Chosen target column:", TARGET_COL)


Chosen modality column: modality
Chosen task column: task
Chosen target column: coarse_affect


In [13]:
if MODALITY_COL:
    modality_values = sorted([str(x) for x in df[MODALITY_COL].dropna().unique().tolist()])
    print("Observed modality labels:")
    print(modality_values)
else:
    print("No modality column detected automatically.")


Observed modality labels:
['audio', 'audiovisual', 'video']


### Scientific check

Before E03 is considered strong, you should know whether the dataset contains exactly the intended modality families.

If the observed labels are noisy, redundant, or unexpected, then E03 should not auto-expand over them without manual approval.


## 4. Sample sufficiency audit by modality


In [14]:
if not MODALITY_COL or not TARGET_COL:
    raise ValueError("Need both a modality column and target column for the E03 audit.")

valid = df.dropna(subset=[MODALITY_COL, TARGET_COL]).copy()
print("Rows with non-missing modality and target:", len(valid))


Rows with non-missing modality and target: 3402


In [15]:
overall_counts = (
    valid.groupby(MODALITY_COL)[TARGET_COL]
    .value_counts()
    .rename("count")
    .reset_index()
    .sort_values([MODALITY_COL, "count"], ascending=[True, False])
)
overall_counts


,modality,coarse_affect,count
0,audio,negative,504
1,audio,positive,504
2,audio,neutral,126
3,audiovisual,negative,504
4,audiovisual,positive,504
5,audiovisual,neutral,126
6,video,negative,504
7,video,positive,504
8,video,neutral,126


In [16]:
subject_counts = (
    valid.groupby(["subject", MODALITY_COL])[TARGET_COL]
    .value_counts()
    .rename("count")
    .reset_index()
    .sort_values(["subject", MODALITY_COL, "count"], ascending=[True, True, False])
)
subject_counts.head(50)


,subject,modality,coarse_affect,count
0,sub-001,audio,negative,228
1,sub-001,audio,positive,228
2,sub-001,audio,neutral,57
3,sub-001,audiovisual,negative,228
4,sub-001,audiovisual,positive,228
5,sub-001,audiovisual,neutral,57
6,sub-001,video,negative,228
7,sub-001,video,positive,228
8,sub-001,video,neutral,57
9,sub-002,audio,negative,276


In [17]:
# Number of unique target classes per subject-modality-session
session_coverage = (
    valid.groupby(["subject", MODALITY_COL, "session"])[TARGET_COL]
    .nunique()
    .rename("n_target_classes")
    .reset_index()
    .sort_values(["subject", MODALITY_COL, "n_target_classes"])
)
session_coverage.head(50)


,subject,modality,session,n_target_classes
0,sub-001,audio,ses-01,3
1,sub-001,audio,ses-02,3
2,sub-001,audio,ses-03,3
3,sub-001,audio,ses-04,3
4,sub-001,audio,ses-05,3
5,sub-001,audio,ses-06,3
6,sub-001,audio,ses-07,3
7,sub-001,audio,ses-08,3
8,sub-001,audio,ses-09,3
9,sub-001,audio,ses-10,3


### Interpretation

You want to know:
- which modalities have enough rows overall
- which modalities have enough rows per subject
- whether subject-modality-session slices look too sparse for LOSO-session interpretation

A modality that is extremely sparse should not be treated as decision-eligible just because the code can run it.


## 5. LOSO-session viability by modality


In [18]:
def loso_viability_by_modality(frame: pd.DataFrame, target_col: str, modality_col: str) -> pd.DataFrame:
    rows = []
    for (subject, modality), sdf in frame.groupby(["subject", modality_col]):
        all_classes = set(sdf[target_col].dropna().unique().tolist())
        sessions = sorted(sdf["session"].dropna().unique().tolist())
        for heldout_session in sessions:
            train = sdf[sdf["session"] != heldout_session]
            test = sdf[sdf["session"] == heldout_session]
            train_classes = set(train[target_col].dropna().unique().tolist())
            test_classes = set(test[target_col].dropna().unique().tolist())
            rows.append({
                "subject": subject,
                "modality": modality,
                "heldout_session": heldout_session,
                "n_train": len(train),
                "n_test": len(test),
                "n_all_classes": len(all_classes),
                "n_train_classes": len(train_classes),
                "n_test_classes": len(test_classes),
                "train_has_all_global_classes": train_classes == all_classes,
                "missing_from_train": sorted(test_classes - train_classes),
            })
    return pd.DataFrame(rows)

loso_modality = loso_viability_by_modality(valid, TARGET_COL, MODALITY_COL)
loso_modality.head(50)


,subject,modality,heldout_session,n_train,n_test,n_all_classes,n_train_classes,n_test_classes,train_has_all_global_classes,missing_from_train
0,sub-001,audio,ses-01,486,27,3,3,3,True,[]
1,sub-001,audio,ses-02,486,27,3,3,3,True,[]
2,sub-001,audio,ses-03,486,27,3,3,3,True,[]
3,sub-001,audio,ses-04,486,27,3,3,3,True,[]
4,sub-001,audio,ses-05,486,27,3,3,3,True,[]
5,sub-001,audio,ses-06,486,27,3,3,3,True,[]
6,sub-001,audio,ses-07,486,27,3,3,3,True,[]
7,sub-001,audio,ses-08,486,27,3,3,3,True,[]
8,sub-001,audio,ses-09,486,27,3,3,3,True,[]
9,sub-001,audio,ses-10,486,27,3,3,3,True,[]


In [19]:
problems = loso_modality[
    (~loso_modality["train_has_all_global_classes"]) |
    (loso_modality["n_train"] == 0) |
    (loso_modality["n_test"] == 0)
].copy()

problems.sort_values(["subject", "modality", "heldout_session"]).head(100)


,subject,modality,heldout_session,n_train,n_test,n_all_classes,n_train_classes,n_test_classes,train_has_all_global_classes,missing_from_train


### Why this matters

A modality-specific comparison is only scientifically useful if modality slices remain viable under the same split logic.

If one modality repeatedly fails LOSO viability, that is not just an engineering nuisance — it changes how much you can trust the comparison.


## 6. Task-composition confound audit

This is one of the most important E03 checks.

If different modalities are associated with different task mixtures, then E03 may partly reflect:
- modality effects
- task effects
- or their interaction

rather than a pure pooling-by-modality decision.


In [20]:
if TASK_COL:
    modality_task = (
        valid.groupby([MODALITY_COL, TASK_COL])
        .size()
        .rename("count")
        .reset_index()
        .sort_values([MODALITY_COL, "count"], ascending=[True, False])
    )
    modality_task
else:
    print("No task column detected automatically, so the modality/task confound audit cannot run.")


In [21]:
if TASK_COL:
    ct = pd.crosstab(valid[MODALITY_COL], valid[TASK_COL])
    ct


### Interpretation

If modalities are strongly imbalanced across tasks, then E03 is scientifically weaker unless you explicitly accept that limitation.

Best case:
- each modality spans similar task families

Risk case:
- one modality is mostly tied to one task family
- another modality is tied to a different one

If that happens, modality-specific performance is not cleanly separable from task composition.


## 7. Readiness verdict helper


In [22]:
readiness_notes = []

# Workbook readiness
if len(e03_wb) == 1:
    row = e03_wb.iloc[0]
    if str(row.get("enabled", "")).strip().lower() != "yes":
        readiness_notes.append("Workbook path not ready: E03 is not enabled.")
    if str(row.get("filter_modality", "")).strip().upper() == "VARIES":
        readiness_notes.append("Workbook path not ready: filter_modality is still VARIES.")
    if pd.isna(row.get("search_space_id")) or str(row.get("search_space_id")).strip() == "":
        readiness_notes.append("Workbook path not ready: no executable expansion/search space is attached.")

# Registry readiness
if e03_reg and e03_reg.get("executable_now") is True:
    readiness_notes.append("Registry path ready: E03 is executable through the registry.")
else:
    readiness_notes.append("Registry path not ready: registry does not mark E03 executable.")

# Dynamic modality expansion note
if e03_reg:
    vt = e03_reg.get("variant_templates", [])
    if any(isinstance(t, dict) and isinstance(t.get("expand"), dict) and "filter_modality" in t["expand"] for t in vt):
        readiness_notes.append("Scientific caution: registry E03 appears to auto-expand modalities from the dataset rather than a fixed approved modality list.")

for note in readiness_notes:
    print("-", note)


- Workbook path not ready: E03 is not enabled.
- Workbook path not ready: filter_modality is still VARIES.
- Workbook path not ready: no executable expansion/search space is attached.
- Registry path ready: E03 is executable through the registry.
- Scientific caution: registry E03 appears to auto-expand modalities from the dataset rather than a fixed approved modality list.


## 8. Recommended modifications to make E03 as strong as possible

The likely recommendations are:

1. **Workbook path**
   - make E03 workbook-executable
   - do not leave `filter_modality = VARIES`

2. **Fix the approved modality set explicitly**
   - do not rely on whatever modality labels exist in the dataset
   - pre-register the exact modality families you will compare

3. **Prefer explicit workbook rows over a search space**
   - pooled baseline is best represented by a blank `filter_modality`
   - modality-specific variants are best represented by explicit modality labels
   - this is cleaner than encoding blank-vs-specific semantics in a search space

4. **Add a modality sufficiency gate**
   - a modality-specific run should not be decision-eligible if it fails sample/class/session viability

5. **Review task composition by modality**
   - if modalities differ strongly in task composition, write that limitation explicitly
   - or control it in a later refinement


## 9. Dry-run command for E03

This command only prints the run command. It does not execute automatically.


In [23]:
dry_run_cmd = [
    sys.executable, "-m", "uv", "run",
    "thesisml-run-decision-support",
    "--registry", str(REGISTRY),
    "--experiment-id", "E03",
    "--index-csv", str(INDEX_CSV),
    "--data-root", str(DATA_ROOT),
    "--cache-dir", str(CACHE_DIR),
    "--output-root", str(REPO_ROOT / "outputs/artifacts/decision_support"),
    "--dry-run",
]
print(" ".join(f'"{part}"' if " " in part else part for part in dry_run_cmd))


d:\Khaled\Projects\VScode\Thesis\.venv-gpu\Scripts\python.exe -m uv run thesisml-run-decision-support --registry D:\Khaled\Projects\VScode\Thesis\configs\decision_support_registry.json --experiment-id E03 --index-csv D:\Khaled\Projects\VScode\Thesis\Data\processed\dataset_index.csv --data-root D:\Khaled\Projects\VScode\Thesis\Data --cache-dir D:\Khaled\Projects\VScode\Thesis\Data\processed\feature_cache --output-root D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support --dry-run


In [25]:
# Uncomment to actually execute the dry-run.
result = subprocess.run(dry_run_cmd, text=True, capture_output=True)
print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)


Return code: 0
Campaign complete
- campaign_id: 20260324_121215_675516
- campaign_root: D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support\campaigns\20260324_121215_675516
- selected_experiments: E03
- status_counts: {'dry_run': 8}
- run_log_export: D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support\campaigns\20260324_121215_675516\run_log_export.csv
- decision_summary: D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support\campaigns\20260324_121215_675516\decision_support_summary.csv
- decision_report: D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support\campaigns\20260324_121215_675516\decision_recommendations.md
- aggregation: D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support\campaigns\20260324_121215_675516\result_aggregation.json
- summary_outputs_export: D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support\campaigns\20260324_121215_675516\summary_outputs_export.csv
- manifest: D:\Khaled

## Final notebook conclusion

The most likely conclusion pattern is:

- **registry path:** operationally ready
- **workbook path:** not ready
- **scientific correctness:** conditional
- **main scientific risk:** uncontrolled modality expansion and modality/task composition confounding

So E03 can probably run now, but to make it as strong as possible you should:
- pre-register the exact modality set
- represent pooled vs modality-specific variants explicitly
- apply sufficiency checks before interpretation
- review task composition before making a final modality-pooling decision
